# 🎓 LangChain LLM API Call + LangSmith Tracing


## What you will do in this notebook
1. Configure an LLM key (OpenAI **or** any compatible provider) and run a **LangChain** chat call.
2. Use a **Prompt Template** with variables (reusable prompts).
3. Enable **LangSmith tracing** to capture inputs/outputs for debugging & classroom review.
4. Run a controlled experiment showing **why naive “just paste all documents” breaks** — motivating **RAG**.

> ✅ In this session we **show the problems** that appear in real applications.  
> We intentionally **do not implement RAG** here — that is the next session.

---

## 0) Install dependencies

> If you are using Google Colab, run this cell once per runtime.

In [3]:
!pip -q install -U langchain langchain-openai langchain-community langchain-core tiktoken langsmith[openai-agents]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.7/325.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 11.7 MB/s eta 0:00:00


---

## 1) API Key setup (Colab-friendly)

### Option A (recommended in Colab): use `userdata`
1. In Colab: **🔑 Secrets** (left panel) → add a secret named `OPENAI_API_KEY`
2. Keep it **private**.

### Option B: environment variable
Set `OPENAI_API_KEY` in your environment.

> If you are using a different provider (Groq/OpenRouter/…),
> you can still use LangChain with the appropriate chat model wrapper.
> For this notebook, we keep it simple and use `ChatOpenAI`.

In [9]:
import os

# Colab: safely load from secrets (if available)
try:
    from google.colab import userdata
    if userdata.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is missing. Add it to Colab Secrets or environment variables."
print("✅ API key detected.")

✅ API key detected.


---

## 2) LangSmith setup (Tracing)

LangSmith helps you **trace** what happened inside a LangChain run:
- Inputs passed to the model
- Prompt formatting
- Outputs from the model
- Timing and errors

### Steps to enable
1. Create a (free) LangSmith account and an API key
2. Put your key into Colab secrets as `LANGSMITH_API_KEY`
3. Set tracing env variables below

> If you don’t have LangSmith set up, you can still run the notebook — tracing will be disabled.

In [10]:
# --- Optional: enable LangSmith tracing ---
# Add LANGSMITH_API_KEY in Colab Secrets for tracing.

try:
    from google.colab import userdata
    langsmith_key = userdata.get("LANGSMITH_API_KEY")
except Exception:
    langsmith_key = os.environ.get("LANGSMITH_API_KEY")

if langsmith_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_key
    os.environ["LANGCHAIN_TRACING"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = "LANGCHAIN_DEMO"
    print("✅ LangSmith tracing is ON (LANGCHAIN_TRACING_V2=true).")
    print("   Project:", os.environ["LANGCHAIN_PROJECT"])
else:
    print("ℹ️ LangSmith tracing is OFF (no LANGSMITH_API_KEY found).")
    print("   You can still proceed; you just won’t see traces in LangSmith.")

✅ LangSmith tracing is ON (LANGCHAIN_TRACING_V2=true).
   Project: LANGCHAIN_DEMO


---

## 3) Load the dataset: `students_details.txt`

This file is our “knowledge base” for the exercise.

✅ We will **only use text** (no PDFs) so everyone can follow easily.

In [11]:
from pathlib import Path

DATA_PATH = Path("/content/students_details.txt")  # provided in this training repo / environment
assert DATA_PATH.exists(), f"File not found: {DATA_PATH}"

students_text = DATA_PATH.read_text(encoding="utf-8")
print("✅ Loaded:", DATA_PATH.name)
print("Characters:", len(students_text))
print("\n--- Preview ---\n")
print(students_text[:400])

✅ Loaded: students_details.txt
Characters: 1539

--- Preview ---

Name: Aisha Rahman
University: Universiti Malaya
Class: AI-2024
Project: Smart Traffic Light Control using YOLOv8
Description: Detects vehicle congestion and adapts signal timing dynamically using real-time video analysis.

Name: Rahul Kumar
University: UTM
Class: AI-2024
Project: Emotion Detection Chatbot
Description: Identifies and responds to user emotions in chat using text embeddings and sent


---

## 4) Minimal LangChain LLM API call (Prompt Template)

We build:

**Prompt Template → Chat Model → Output Parser**

And we will ask questions strictly from the file content.

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL_NAME = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL_NAME, temperature=0.0)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a teaching assistant. "
     "Answer ONLY using the provided CONTEXT. "
     "If the answer is not explicitly present, reply exactly: NOT FOUND IN PROVIDED CONTEXT."),
    ("human",
     "CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\n"
     "Output rules:\n- Keep it short\n- If listing, use bullet points")
])

chain = qa_prompt | llm | StrOutputParser()

question = "List all student names in the file."
print(chain.invoke({"context": students_text, "question": question}))

- Aisha Rahman: Smart Traffic Light Control using YOLOv8
- Rahul Kumar: Emotion Detection Chatbot


---

## 5) Exercise for participants (edit the question)

Try questions like:
- “Which student is from UTM?”
- “Which project uses YOLOv8?”
- “Give the project title of Noor Fatima”
- “How many students are listed?” *(Hint: this may require careful prompting)*

✅ Your task: change `my_question` and run the cell.

---

# 6) Challenges of the current approach (leading to RAG)

In real systems, the file becomes:
- many files (reports, meeting notes, tickets)
- larger text (10×, 100×)
- updated often

The naïve approach is: **paste all text into the prompt** and ask the model.

We demonstrate 3 problems:
1. **Context window / token limits**
2. **Cost & latency grows with text size**
3. **Reliability drops when content is truncated**

> We show the problems now.  
> We solve them in the next session using **RAG**.

## 6.1 Token counting (why “paste everything” doesn’t scale)

We estimate tokens using `tiktoken` (good enough for classroom intuition).

In [17]:
import tiktoken

def count_tokens(text: str) -> int:
    enc = tiktoken.get_encoding("o200k_base")
    return len(enc.encode(text))

base_tokens = count_tokens(students_text)
print("Approx tokens in students_details.txt:", base_tokens)

# Simulate growth (e.g., many files / many semesters)
grown_text = (students_text + "\n\n") * 30
grown_tokens = count_tokens(grown_text)

print("Approx tokens after 30x growth:", grown_tokens)

# Demo threshold (not necessarily the true model limit)
THRESHOLD = 2500
if grown_tokens > THRESHOLD:
    print(f"⚠️ Exceeds demo threshold ({THRESHOLD}) → truncation / cost / latency becomes significant.")
else:
    print("✅ Still within demo threshold (for now).")

Approx tokens in students_details.txt: 331
Approx tokens after 30x growth: 9930
⚠️ Exceeds demo threshold (2500) → truncation / cost / latency becomes significant.


## 6.2 Reliability issue: truncation removes the evidence

We create a “needle-in-haystack” situation:
- We append a single *unique fact* at the very end.
- Then we truncate the text (a common shortcut in apps).
- The model can no longer answer correctly, because the evidence is gone.

In [18]:
needle = "\nUNIQUE FACT: The student token for demo is DEMO-TOKEN-9273 (stored only at the end).\n"
haystack = grown_text + needle

print("Does full text contain the UNIQUE FACT?", "DEMO-TOKEN-9273" in haystack)

q = "What is the demo student token?"

# Full context
full_answer = chain.invoke({"context": haystack, "question": q})
print("FULL context answer:\n", full_answer)

# Naive truncation (keep only first N characters)
N = 2000
truncated = haystack[:N]
print("\nDoes TRUNCATED text contain the UNIQUE FACT?", "DEMO-TOKEN-9273" in truncated)

trunc_answer = chain.invoke({"context": truncated, "question": q})
print("\nTRUNCATED context answer:\n", trunc_answer)

Does full text contain the UNIQUE FACT? True
FULL context answer:
 - DEMO-TOKEN-9273

Does TRUNCATED text contain the UNIQUE FACT? False

TRUNCATED context answer:
 NOT FOUND IN PROVIDED CONTEXT.


---

# 7) What this implies (bridge to next session: RAG)

From the experiments:
- “Stuff all text into prompt” **does not scale**
- Truncation **removes the exact facts**
- Reliability can drop, especially with more creativity

✅ Next session: **RAG**
- chunk the documents
- index them (embeddings / vector store)
- retrieve only relevant chunks for the question
- answer grounded in retrieved evidence

> Today: you saw the “pain” clearly.  
> Next: we solve it with RAG.